# Workflow Optimizer — V0

**Goal.** Given a *task* — described by a seed prompt and/or a labeled dataset — automatically find the LLM workflow that maximizes accuracy under a cost ceiling (or minimizes cost above an accuracy floor).

**How V0 works.**
1. **Profile the task** — from your seed prompt (and a few examples, if given), the model infers a task description *and the answer format*: how to extract the answer and how to score it (numeric tolerance / exact-match / LLM-judge). If you didn't supply a dataset, it generates one.
2. **Design workflows** — a Claude Code agent (web search + Bash) writes candidate `solve(question, llm)` **programs** and self-tests them on a dev split. The workflow is arbitrary code, so it can be *any* paradigm.
3. **Search** — run each program on the held-out test split through a single metered `llm(...)` runtime, and pick the best point on the accuracy/cost **Pareto frontier**.

**The fixed contract — everything general rides on this:**
- `solve(question, llm) -> answer` — the only shape the harness knows.
- `llm(prompt, max_tokens=…, model=…)` — the single instrumented, budget-capped call site.
- a task-inferred `extract` + `check` — the paradigm-blind evaluator (numeric / exact / LLM-judge).

**Setup.** Real Anthropic API calls throughout — set `ANTHROPIC_API_KEY`. Requires `claude-agent-sdk`. Generated programs run in a restricted namespace with hard caps — a guardrail, **not** a security sandbox.

**Bonus:** a surrogate predictor that learns to predict accuracy & cost, so you can skip most rollouts.

In [24]:
import os, re, json, statistics

import anthropic
import pandas as pd
import matplotlib.pyplot as plt

# ---- Pricing: USD per 1,000,000 tokens (input, output) --------------------
PRICES = {
    "claude-haiku-4-5": (1.0,  5.0),
    "claude-sonnet-5":  (3.0, 15.0),
    "claude-opus-4-8":  (5.0, 25.0),
}

def cost_usd(model, tokens_in, tokens_out):
    pin, pout = PRICES[model]
    return (tokens_in * pin + tokens_out * pout) / 1_000_000

# ---- Anthropic API client -------------------------------------------------
# Every rollout is a real API call, so a key is required.
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in your shell env before running.")
client = anthropic.Anthropic()

print("Anthropic client ready.")

Anthropic client ready.


In [25]:
# ---- Define the problem (the ONLY per-task input) -------------------------
# Describe the task in SEED_PROMPT. Optionally supply your own labeled DATASET
# as a list of {"question": <input str>, "answer": <target>}; leave it None to
# have one generated. The answer format (numeric / exact-match / free-form) is
# figured out automatically in the profiling step below.
SEED_PROMPT = ("Solve multi-step grade-school math word problems. "
               "Each answer is a single integer.")
DATASET = None      # or: [{"question": "...", "answer": 42}, ...]

print("Seed prompt set." +
      ("  (dataset provided)" if DATASET else "  (dataset will be generated)"))

Seed prompt set.  (dataset will be generated)


In [26]:
# ---- Models to search over ------------------------------------------------
# The optimizer searches over (model x strategy). The strategies are NOT written
# here — the model designs them further down. Add "claude-opus-4-8" to include Opus.
MODELS = ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-4-8"]
print("Models:", ", ".join(MODELS))

Models: claude-haiku-4-5, claude-sonnet-5, claude-opus-4-8


In [27]:
# ---- One real API call ----------------------------------------------------
def _call_api(model, prompt, max_tokens):
    # Single real API call -> (text, tokens_in, tokens_out).
    # thinking=disabled so the *strategy* is the only reasoning knob (otherwise
    # newer models reason by default, making a "direct" answer neither direct nor
    # cheap). No `temperature` (Sonnet 5 / Opus 4.8 reject it); sampling diversity
    # for majority-vote strategies comes from the model's default sampling.
    msg = client.messages.create(
        model=model, max_tokens=max_tokens,
        thinking={"type": "disabled"},
        messages=[{"role": "user", "content": prompt}],
    )
    text = "".join(b.text for b in msg.content if b.type == "text")
    return text, msg.usage.input_tokens, msg.usage.output_tokens

In [28]:
# ---- Answer parsing + correctness, from a serializable spec ---------------
def extract_number(text):
    # General helper (also injected into program sandboxes): last number in text.
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text or "")
    if not nums:
        return None
    try:
        return float(nums[-1].replace(",", ""))
    except ValueError:
        return None

def make_extractor(spec):
    t = spec.get("type", "full")
    if t == "last_number":
        def ex(text):
            n = extract_number(text)
            return "" if n is None else (str(int(n)) if n == int(n) else str(n))
        return ex
    if t == "last_line":
        def ex(text):
            lines = [ln for ln in (text or "").strip().splitlines() if ln.strip()]
            return lines[-1].strip() if lines else ""
        return ex
    return lambda text: (text or "").strip()          # "full"

def make_checker(spec, judge_call):
    t = spec.get("type", "exact")
    if t == "numeric":
        tol = float(spec.get("tol", 1e-6))
        def _num(x):                                   # tolerate bare OR prose values
            s = str(x).replace(",", "").strip()
            try:
                return float(s)
            except ValueError:
                return extract_number(s)
        def ck(pred, gold):
            pn, gn = _num(pred), _num(gold)
            return pn is not None and gn is not None and abs(pn - gn) <= tol
        return ck
    if t == "exact":
        def norm(x):
            s = str(x)
            if spec.get("strip", True):
                s = s.strip()
            if spec.get("casefold", True):
                s = s.casefold()
            return s
        return lambda pred, gold: norm(pred) == norm(gold)
    # llm_judge — a cheap model decides equivalence (judge cost is the
    # evaluator's, never counted as workflow cost).
    jm = spec.get("model", "claude-haiku-4-5")
    def ck(pred, gold):
        q = (f"Task: {spec.get('task', '')}\nReference answer: {gold}\n"
             f"Candidate answer: {pred}\nIs the candidate correct / equivalent to "
             "the reference? Answer 'yes' or 'no'.")
        return judge_call(jm, q).strip().lower().startswith("y")
    return ck

In [29]:
# ---- Profile the task: infer description + answer format; get a dataset ----
PROFILER_MODEL = "claude-opus-4-8"

TASK_PROFILE_SCHEMA = {
    "type": "object",
    "properties": {
        "description": {"type": "string"},
        "answer_type": {"type": "string"},
        "extract": {"type": "object", "properties": {
            "type": {"type": "string", "enum": ["last_number", "last_line", "full"]}},
            "required": ["type"], "additionalProperties": False},
        "check": {"type": "object", "properties": {
            "type": {"type": "string", "enum": ["numeric", "exact", "llm_judge"]},
            "tol": {"type": "number"}, "casefold": {"type": "boolean"},
            "strip": {"type": "boolean"}},
            "required": ["type"], "additionalProperties": False},
    },
    "required": ["description", "answer_type", "extract", "check"],
    "additionalProperties": False,
}

def _json_call(model, prompt, schema, max_tokens=2048):
    msg = client.messages.create(
        model=model, max_tokens=max_tokens, thinking={"type": "disabled"},
        output_config={"format": {"type": "json_schema", "schema": schema}},
        messages=[{"role": "user", "content": prompt}])
    return json.loads("".join(b.text for b in msg.content if b.type == "text"))

def generate_dataset(description, n=12):
    schema = {"type": "object", "properties": {"data": {"type": "array", "items": {
        "type": "object", "properties": {
            "question": {"type": "string"}, "answer": {"type": "string"}},
        "required": ["question", "answer"], "additionalProperties": False}}},
        "required": ["data"], "additionalProperties": False}
    p = (f"Generate {n} diverse labeled examples for this task:\n{description}\n\n"
         "Return objects with 'question' (the input) and 'answer' (the correct final "
         "target ONLY — a bare value such as the number or the label, with NO "
         "explanation, units, or extra words). Make them solvable and unambiguous.")
    return _json_call(PROFILER_MODEL, p, schema)["data"]

def profile_task(seed_prompt, dataset):
    sample = ""
    if dataset:
        sample = "\n\nExample (input -> target) pairs:\n" + "\n".join(
            f"- {d['question']!r} -> {d['answer']!r}" for d in dataset[:5])
    p = (f"A user wants to build an LLM workflow for this task:\n{seed_prompt}{sample}\n\n"
         "Return: (1) a crisp one-paragraph 'description' for a workflow designer; "
         "(2) 'answer_type' (short label); (3) an 'extract' spec = how to pull the "
         "answer from a model's raw text output (last_number for numeric answers, "
         "last_line for a short label on its own line, full otherwise); (4) a 'check' "
         "spec = how to score correctness (numeric with a tolerance for numbers, exact "
         "for labels/short strings, llm_judge for free-form / semantic answers).")
    return _json_call(PROFILER_MODEL, p, TASK_PROFILE_SCHEMA)

profile = profile_task(SEED_PROMPT, DATASET)
DATA = DATASET if DATASET else generate_dataset(profile["description"])

profile["check"]["task"] = profile["description"]      # context for an llm_judge
EXTRACT = make_extractor(profile["extract"])
CHECK   = make_checker(profile["check"], lambda m, pr: _call_api(m, pr, 16)[0])

_cut = max(1, len(DATA) * 3 // 5)
DATA_DEV, DATA_TEST = DATA[:_cut], DATA[_cut:]

print(f"answer_type = {profile['answer_type']!r}   "
      f"extract = {profile['extract']['type']}   check = {profile['check']['type']}")
print(f"{len(DATA)} examples  ->  {len(DATA_DEV)} dev / {len(DATA_TEST)} test\n")
print("Description:", profile["description"])

answer_type = 'integer'   extract = last_number   check = numeric
12 examples  ->  7 dev / 5 test

Description: Build a workflow that solves multi-step grade-school math word problems. The model should read the problem, reason step-by-step through the necessary arithmetic operations, and arrive at a final answer that is always a single integer. Prompt the model to show its reasoning and then clearly state the final numeric answer, ideally on its own line or at the end of the response so it can be reliably extracted and scored against the expected integer.


In [35]:
# ---- Pareto frontier + constrained selection ------------------------------
def pareto_front(results):
    # Configs where no other config is both cheaper AND at least as accurate.
    front = []
    for r in results:
        dominated = any(
            o is not r and o["cost_per_query"] <= r["cost_per_query"]
            and o["accuracy"] >= r["accuracy"]
            and (o["cost_per_query"] < r["cost_per_query"] or o["accuracy"] > r["accuracy"])
            for o in results
        )
        if not dominated:
            front.append(r)
    return sorted(front, key=lambda r: r["cost_per_query"])

def best_under_budget(results, max_cost_per_query):
    ok = [r for r in results if r["cost_per_query"] <= max_cost_per_query]
    return max(ok, key=lambda r: r["accuracy"]) if ok else None

def cheapest_above_accuracy(results, min_accuracy):
    ok = [r for r in results if r["accuracy"] >= min_accuracy]
    return min(ok, key=lambda r: r["cost_per_query"]) if ok else None

## The proposer is a Claude Code agent driven by two skills

The proposer is a **Claude Agent SDK** agent with **web search + Bash + file
tools**. Its methodology and its dev evaluator are packaged as standard
`SKILL.md` skills under `skills/`:

- **workflow-design** — the design methodology + the `solve(question, llm)` contract.
- **workflow-eval** — bundles `eval_candidate.py`, which scores a candidate on the
  dev set (reconstructing the task's extract/check from `task_spec.json`).

We copy both into the agent's `.claude/skills/` so the SDK discovers them
(`setting_sources=["project"]`, `skills=[...]`, and `"Skill"` in `allowed_tools`).
The agent researches techniques, writes candidate programs, tests each with the
workflow-eval skill, and writes its picks to `programs.json` — read back as
`PROGRAMS`. Candidate programs still run only through our metered, capped,
sandboxed evaluator (on DEV here, and on the held-out TEST split in the search).

> Requires `claude-agent-sdk` + `ANTHROPIC_API_KEY`; the agent's tool calls and
> self-tests make real API calls (spends money).

In [ ]:
# ---- Agentic proposer: a Claude Code agent driven by two skills ------------
# The design methodology + the dev evaluator are standard SKILL.md skills under
# ./skills/ (workflow-design, workflow-eval), copied into the agent's
# .claude/skills/ so the SDK discovers them. The agent designs + self-tests
# candidate programs and writes its picks to programs.json.
#
# We run the agent in a CLEAN SUBPROCESS (run_proposer.py) rather than in-kernel:
# a prior nest_asyncio.apply() monkeypatches asyncio for the whole Jupyter kernel
# session, which breaks asyncio.run even from a worker thread. A subprocess sidesteps
# that entirely. If the agent still doesn't write programs.json (e.g. a dropped
# connection), we salvage the candidate .py files it wrote.
import tempfile, pathlib, shutil, subprocess, sys

PROPOSER_MODEL = "claude-opus-4-8"

AGENT_DIR = pathlib.Path(tempfile.mkdtemp(prefix="proposer_"))
(AGENT_DIR / "task_spec.json").write_text(json.dumps(profile))
(AGENT_DIR / "dev_task.json").write_text(json.dumps(DATA_DEV[:5]))   # small dev = fast self-tests
_skills_dst = AGENT_DIR / ".claude" / "skills"
_skills_dst.mkdir(parents=True)
for _name in ("workflow-design", "workflow-eval"):
    shutil.copytree(pathlib.Path("skills") / _name, _skills_dst / _name)

AGENT_PROMPT = (
    "Design inference-time LLM workflows for this task:\n"
    + profile["description"] + "\n\n"
    "Available models, cheap -> expensive: " + ", ".join(MODELS) + ".\n"
    "Returned answers are scored by extract='" + profile["extract"]["type"] +
    "' then check='" + profile["check"]["type"] + "'.\n\n"
    "Use the `workflow-design` skill for the methodology and program contract, and "
    "the `workflow-eval` skill to test each candidate on the dev set. Write candidate "
    "files and programs.json in your current working directory. Keep 4-5 diverse, "
    "working candidates spanning cheap -> accurate, then write programs.json and stop."
)
(AGENT_DIR / "proposer_config.json").write_text(json.dumps({
    "model": PROPOSER_MODEL,
    "skills": ["workflow-design", "workflow-eval"],
    "allowed_tools": ["WebSearch", "WebFetch", "Bash", "Read", "Write", "Skill"],
    "prompt": AGENT_PROMPT,
}))

# Stream the agent's log live from the subprocess.
_runner = str(pathlib.Path("run_proposer.py").resolve())
_proc = subprocess.Popen([sys.executable, _runner], cwd=str(AGENT_DIR),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="")
_proc.wait()

# ---- collect programs (salvage candidate files if programs.json is missing) --
pj = AGENT_DIR / "programs.json"
if pj.exists():
    raw = json.loads(pj.read_text())
    PROGRAMS = raw["programs"] if isinstance(raw, dict) else raw
else:
    print(f"\nprograms.json missing — salvaging candidate files from {AGENT_DIR}")
    PROGRAMS = [{"name": f.stem, "description": "(recovered from disk)",
                 "code": f.read_text()}
                for f in sorted(AGENT_DIR.glob("*.py"))
                if f.name not in ("run_proposer.py", "eval_candidate.py")
                and "def solve" in f.read_text()]

PROGRAMS = [p for p in PROGRAMS if p.get("code")]
if not PROGRAMS:
    raise RuntimeError(f"no candidate programs found in {AGENT_DIR} — see the log above")

print(f"\n{len(PROGRAMS)} programs proposed (agentic):")
for p in PROGRAMS:
    print(f"  {p['name']:26s} — {p.get('description', '')}")

In [ ]:
# ---- Metered, capped runtime + sandboxed execution of a program -----------
import builtins, signal
from collections import Counter

class BudgetError(RuntimeError):
    pass

class Runtime:
    """The single instrumented call site. Every model call is metered here and
    the per-query budget is enforced here, so cost accounting is paradigm-blind."""
    def __init__(self, default_model, max_calls=24, max_tokens=120_000):
        self.default_model = default_model
        self.max_calls, self.max_tokens = max_calls, max_tokens
        self.calls = self.tokens = 0
        self.cost = 0.0

    def llm(self, prompt, max_tokens=256, model=None):
        if self.calls >= self.max_calls:
            raise BudgetError(f"exceeded {self.max_calls} model calls")
        self.calls += 1
        m = model if model in PRICES else self.default_model     # program may route
        text, ti, to = _call_api(m, str(prompt), int(max_tokens))
        self.tokens += ti + to
        self.cost   += cost_usd(m, ti, to)
        if self.tokens > self.max_tokens:
            raise BudgetError(f"exceeded {self.max_tokens} tokens")
        return text

# ---- Lightweight guardrail for running model-written code ------------------
# A restricted namespace is a GUARDRAIL, not a security boundary. For untrusted
# code use a container/subprocess with real isolation.
_ALLOWED_IMPORTS = {"re", "json", "math", "statistics", "collections",
                    "itertools", "functools", "string"}
def _safe_import(name, *a, **k):
    if name.split(".")[0] not in _ALLOWED_IMPORTS:
        raise ImportError(f"import of '{name}' is not allowed")
    return builtins.__import__(name, *a, **k)

_SAFE_BUILTINS = {n: getattr(builtins, n) for n in (
    "range len min max sum sorted abs round divmod pow enumerate zip map filter "
    "list dict set tuple str int float bool any all isinstance reversed print "
    "Exception ValueError ZeroDivisionError").split()}
_SAFE_BUILTINS["__import__"] = _safe_import

def compile_solve(code):
    ns = {"__builtins__": _SAFE_BUILTINS, "re": re, "json": json,
          "statistics": statistics, "Counter": Counter,
          "extract_number": extract_number, "MODELS": MODELS}
    exec(code, ns)
    solve = ns.get("solve")
    if not callable(solve):
        raise ValueError("program does not define solve(question, llm)")
    return solve

# ---- Wall-clock guard (Unix main thread) for runaway CPU loops -------------
class _Timeout(Exception):
    pass

def _with_timeout(fn, seconds):
    if not hasattr(signal, "SIGALRM"):
        return fn()
    def _h(signum, frame):
        raise _Timeout()
    old = signal.signal(signal.SIGALRM, _h)
    signal.setitimer(signal.ITIMER_REAL, seconds)
    try:
        return fn()
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old)

# ---- Run one program across the dataset, scored by the task's extract/check -
def evaluate_program(program, dataset, default_model, extract, check,
                     per_query_timeout=120):
    try:
        solve = compile_solve(program["code"])
    except Exception as e:                          # didn't even compile
        return {"config": program["name"], "model": default_model,
                "strategy": program["name"], "accuracy": 0.0,
                "cost_per_query": 0.0, "error": f"compile: {e}", "records": []}
    records = []
    for item in dataset:
        rt = Runtime(default_model)
        try:
            ans = _with_timeout(lambda: solve(item["question"], rt.llm),
                                per_query_timeout)
            pred = extract(ans if isinstance(ans, str) else str(ans))
            correct = bool(check(pred, item["answer"]))
        except Exception:                           # budget / timeout / bug -> wrong
            correct = False
        records.append({"model": default_model, "strategy": program["name"],
                        "q_len": len(item["question"]), "correct": int(correct),
                        "cost": rt.cost})
    return {"config": program["name"], "model": default_model,
            "strategy": program["name"],
            "accuracy": sum(r["correct"] for r in records) / len(records),
            "cost_per_query": sum(r["cost"] for r in records) / len(records),
            "records": records}

In [ ]:
# ---- Run every proposed program over the HELD-OUT TEST split --------------
# No static estimate is possible (arbitrary control flow) — we RUN and measure,
# with the runtime's hard caps bounding worst-case spend. Scored with the task's
# inferred EXTRACT + CHECK, on TEST (not the DEV split the proposer tuned on).
BASE_MODEL = MODELS[0]      # default model; programs may escalate to pricier ones
code_results = [evaluate_program(p, DATA_TEST, BASE_MODEL, EXTRACT, CHECK)
                for p in PROGRAMS]

code_df = pd.DataFrame([
    {"program": r["strategy"], "accuracy": r["accuracy"],
     "cost_per_query": r["cost_per_query"], "error": r.get("error", "")}
    for r in code_results
]).sort_values(["accuracy", "cost_per_query"],
               ascending=[False, True]).reset_index(drop=True)
code_df

In [ ]:
# ---- Pareto frontier + constrained pick over the programs -----------------
code_front = pareto_front(code_results)
print("Pareto frontier (cheapest -> best):")
for r in code_front:
    print(f"  {str(r['config']):26s}  acc={r['accuracy']:.2f}  ${r['cost_per_query']:.5f}/query")

BUDGET = 0.002
pick = best_under_budget(code_results, BUDGET)
print(f"\nBest program under ${BUDGET}/query:  {pick['config']}"
      f"  (acc={pick['accuracy']:.2f}, ${pick['cost_per_query']:.5f})"
      if pick else "\nNo program under budget.")

TARGET = 0.80
pick2 = cheapest_above_accuracy(code_results, TARGET)
print(f"Cheapest program with acc >= {TARGET}:  {pick2['config']}"
      f"  (acc={pick2['accuracy']:.2f}, ${pick2['cost_per_query']:.5f})"
      if pick2 else f"No program reaches acc >= {TARGET}.")

fig, ax = plt.subplots(figsize=(7, 5))
for r in code_results:
    ax.scatter(r["cost_per_query"], r["accuracy"], s=60, color="#888888")
    ax.annotate(str(r["config"]), (r["cost_per_query"], r["accuracy"]),
                fontsize=8, xytext=(5, 4), textcoords="offset points")
ax.plot([r["cost_per_query"] for r in code_front], [r["accuracy"] for r in code_front],
        "-o", color="#d9534f", label="Pareto frontier")
ax.set_xlabel("cost per query (USD)"); ax.set_ylabel("accuracy")
ax.set_title("LLM-written workflows: accuracy vs. cost"); ax.legend()
plt.tight_layout(); plt.show()

## Bonus 1 — Surrogate predictor (skip the rollouts)

Rollouts cost money and time. Idea: train a small model to predict **accuracy** and **cost** from `(query, program)`, then score workflows *without running them*. Here we reuse the per-example rollouts we already collected as training data. In real use you would train on a small subset of `(query, program)` rollouts and predict the rest — saving the majority of the rollout budget.

In [ ]:
# ---- Train surrogate models on the rollouts we already collected ----------
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

RECORDS   = [rec for r in code_results for rec in r["records"]]
PROG_NAMES = [p["name"] for p in PROGRAMS]
PROG_IDX   = {p: i for i, p in enumerate(PROG_NAMES)}

def featurize(program, q_len):
    f = [0] * len(PROG_NAMES) + [q_len]
    f[PROG_IDX[program]] = 1
    return f

X      = np.array([featurize(r["strategy"], r["q_len"]) for r in RECORDS])
y_acc  = np.array([r["correct"] for r in RECORDS])
y_cost = np.array([r["cost"] for r in RECORDS])

acc_model  = RandomForestClassifier(n_estimators=100, random_state=0).fit(X, y_acc)
cost_model = RandomForestRegressor(n_estimators=100, random_state=0).fit(X, y_cost)

def predict_accuracy(feats):
    # P(correct), robust to the case where every rollout was the same class.
    classes = list(acc_model.classes_)
    proba = acc_model.predict_proba(feats)
    return proba[:, classes.index(1)].mean() if 1 in classes else 0.0

# Predict each program's accuracy/cost WITHOUT new rollouts; compare to measured.
print(f"{'program':26s} {'pred_acc':>8s} {'true_acc':>8s} {'pred_$/q':>10s} {'true_$/q':>10s}")
for r in code_results:
    feats = np.array([featurize(r["strategy"], len(it["question"])) for it in DATA_TEST])
    pred_acc  = predict_accuracy(feats)
    pred_cost = cost_model.predict(feats).mean()
    print(f"{str(r['config']):26s} {pred_acc:8.2f} {r['accuracy']:8.2f} "
          f"{pred_cost:10.5f} {r['cost_per_query']:10.5f}")